In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("USAccidents_LocalCSV") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark started")

Spark started


In [4]:
import os
url = "https://huggingface.co/datasets/nateraw/us-accidents/resolve/main/US_Accidents_Dec21_updated.csv"
output_path = "/tmp/us_accidents.csv"
try:
    os.system(f"wget -O {output_path} {url}")
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(output_path)
    print("Data loaded successfully")
except Exception as e:
    print("Error during data ingestion:", e)
df.show(5)

Data loaded successfully
+---+--------+-------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+--------------------+------+-----------+----+----------+----------+-----+-------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|           End_Lat|           End_Lng|       Distance(mi)|         Description|Number|     Street|Side|      City|    County|State|Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Di

In [5]:
print("Checking schema and data types")
df.printSchema()

print("Checking duplicate rows")
print("Duplicates:", df.count() - df.dropDuplicates().count())

print("Checking invalid values")
df.select("Severity").groupBy("Severity").count().show()

Checking schema and data types
root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Number: double (nullable = true)
 |-- Street: string (nullable = true)
 |-- Side: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (

In [6]:
from pyspark.sql import functions as F
total_rows = df.count()
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).collect()[0].asDict()
print("Missing Values:")
for col, cnt in sorted(null_counts.items(), key=lambda x: x[1], reverse=True):
    if cnt > 0:
        pct = round(cnt / total_rows * 100, 2)
        print(f"{col}: {cnt} ({pct}%)")

Missing Values:
Number: 1743911 (61.29%)
Precipitation(in): 549458 (19.31%)
Wind_Chill(F): 469643 (16.51%)
Wind_Speed(mph): 157944 (5.55%)
Wind_Direction: 73775 (2.59%)
Humidity(%): 73092 (2.57%)
Weather_Condition: 70636 (2.48%)
Visibility(mi): 70546 (2.48%)
Temperature(F): 69274 (2.43%)
Pressure(in): 59200 (2.08%)
Weather_Timestamp: 50736 (1.78%)
Airport_Code: 9549 (0.34%)
Timezone: 3659 (0.13%)
Sunrise_Sunset: 2867 (0.1%)
Civil_Twilight: 2867 (0.1%)
Nautical_Twilight: 2867 (0.1%)
Astronomical_Twilight: 2867 (0.1%)
Zipcode: 1319 (0.05%)
City: 137 (0.0%)
Street: 2 (0.0%)


In [7]:
df = df.drop("Number")
print("Rows:", df.count())
print("Columns:", len(df.columns))
df.show(5)

Rows: 2845342
Columns: 46
+---+--------+-------------------+-------------------+------------------+------------------+------------------+------------------+-------------------+--------------------+-----------+----+----------+----------+-----+-------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|           End_Lat|           End_Lng|       Distance(mi)|         Description|     Street|Side|      City|    County|State|Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_

In [8]:
df = df.drop("Wind_Chill(F)")
print(" Dropped Wind_Chill")

 Dropped Wind_Chill


In [9]:
from pyspark.sql import functions as F
df = df.fillna({
    "Wind_Speed(mph)": 0,
    "Visibility(mi)": df.select(F.avg("Visibility(mi)")).first()[0],
    "Pressure(in)": df.select(F.avg("Pressure(in)")).first()[0]
})
print("Filled numeric missing values")

Filled numeric missing values


In [10]:
df = df.fillna({
    "Weather_Condition": "Unknown"
})
print("Filled categorical missing values")

Filled categorical missing values


In [11]:
df = df.dropna()
print("Dropped remaining null rows")
print("Final Rows:", df.count())

Dropped remaining null rows
Final Rows: 2260486


In [12]:
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).collect()[0].asDict()

print("Remaining null columns:", [c for c, v in null_counts.items() if v > 0])

Remaining null columns: []


In [13]:
print("FINAL DATA SUMMARY\n")
row_count = df.count()
print("Total Rows:", row_count)
col_count = len(df.columns)
print("Total Columns:", col_count)
print("\nSchema:")
df.printSchema()

FINAL DATA SUMMARY

Total Rows: 2260486
Total Columns: 45

Schema:
root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- Side: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nul

In [14]:
from pyspark.sql.functions import to_timestamp
df = df.withColumn("Start_Time", to_timestamp("Start_Time")) \
       .withColumn("End_Time", to_timestamp("End_Time"))

In [15]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

Train rows: 1808797
Test rows: 451689


In [16]:
train_df, val_df = train_df.randomSplit([0.8, 0.2], seed=42)
print("Train rows:", train_df.count())
print("Validation rows:", val_df.count())

Train rows: 1447502
Validation rows: 361295


In [17]:
train_df = train_df.repartition(8, "State")
val_df   = val_df.repartition(8, "State")
test_df  = test_df.repartition(8, "State")
print("Repartitioned all datasets")

Repartitioned all datasets


In [18]:
import os
os.makedirs("data/processed/train", exist_ok=True)
os.makedirs("data/processed/val", exist_ok=True)
os.makedirs("data/processed/test", exist_ok=True)

train_df.write.mode("overwrite").parquet("data/processed/train/")
val_df.write.mode("overwrite").parquet("data/processed/val/")
test_df.write.mode("overwrite").parquet("data/processed/test/")
print("Saved train, validation, and test datasets in Parquet format")

Saved train, validation, and test datasets in Parquet format


In [19]:
train_check = spark.read.parquet("data/processed/train/")
test_check  = spark.read.parquet("data/processed/test/")
print("Train count:", train_check.count())
print("Test count:", test_check.count())

Train count: 1447502
Test count: 451689


In [20]:
spark.read.parquet("data/processed/train/").count()
spark.read.parquet("data/processed/val/").count()
spark.read.parquet("data/processed/test/").count()

451689

In [21]:
print("Train:", spark.read.parquet("data/processed/train/").count())
print("Validation:", spark.read.parquet("data/processed/val/").count())
print("Test:", spark.read.parquet("data/processed/test/").count())

Train: 1447502
Validation: 361295
Test: 451689


In [22]:
import shutil
shutil.make_archive("us_accidents_parquet", 'zip', "data/processed")

'/content/us_accidents_parquet.zip'

In [23]:
from google.colab import files
files.download("us_accidents_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/tmp/us_accidents.csv")

In [27]:
severity_dist = df.groupBy("Severity").count().orderBy("Severity")
severity_dist.toPandas().to_csv("/content/severity_distribution.csv", index=False)
print("Saved: severity_distribution.csv")

Saved: severity_distribution.csv
